# Bag of Words Embeddings
### *The Hundred-Page Language Models Book*, Andriy Burkov
### Section 2.1

In [1]:
import re, torch, torch.nn as nn
torch.manual_seed(42)

In [2]:
# define some text and labels

docs = [
    "movies are fun for everyone",
        "watching movies is great fun.",
        "enjoy a great movie today",
        "research is interesting and important",
        "learning maths is very important",
        "science discovery is interesting",
        "rock is great to listen to",
        "listen to music for fun",
        "music is fun for everyone",
        "listen to folk music!",
        "maths maths maths"
        ]

labels=[1,1,1,3,3,3,2,2,2,2,3]
# 1. movies
# 2. science
# 3. music

# check that we have the correct number of labels
print(len(docs) == len(labels))

num_classes=len(labels)
print(num_classes)

True
11


In [3]:
def tokenize(text):
  return re.findall(r"\w+", text.lower())

def get_vocabulary(texts):
  tokens={token for text in texts for token in tokenize(text)}
  return {word: idx for idx, word in enumerate(sorted(tokens))}

vocabulary=get_vocabulary(docs)
print(type(vocabulary))
print(len(vocabulary))
print(vocabulary)

<class 'dict'>
26
{'a': 0, 'and': 1, 'are': 2, 'discovery': 3, 'enjoy': 4, 'everyone': 5, 'folk': 6, 'for': 7, 'fun': 8, 'great': 9, 'important': 10, 'interesting': 11, 'is': 12, 'learning': 13, 'listen': 14, 'maths': 15, 'movie': 16, 'movies': 17, 'music': 18, 'research': 19, 'rock': 20, 'science': 21, 'to': 22, 'today': 23, 'very': 24, 'watching': 25}


In [4]:
this_text = "Tokenize this text, please!"
print(this_text)
print(tokenize(this_text))

Tokenize this text, please!
['tokenize', 'this', 'text', 'please']


In [5]:
tokens={token for text in docs for token in tokenize(text)}
print(type(tokens))
print(len(tokens))
print(tokens)

<class 'set'>
26
{'important', 'rock', 'folk', 'interesting', 'maths', 'movies', 'everyone', 'and', 'discovery', 'is', 'for', 'music', 'today', 'are', 'research', 'to', 'great', 'enjoy', 'a', 'fun', 'learning', 'science', 'watching', 'very', 'movie', 'listen'}


In [6]:
# convert to Bag of Words
def doc_to_bow(doc, vocabulary):
  """
  inputs:
      doc : str
      vocabulary : dict
  outputs:
      bow : list
  """
  tokens = set(tokenize(doc))
  bow = [0]*len(vocabulary)
  for token in tokens:
    if token in vocabulary:
      bow[vocabulary[token]]+=1
  return bow

In [7]:
vectors=torch.tensor(
    [doc_to_bow(doc,vocabulary) for doc in docs],
    dtype=torch.float32
)

labels=torch.tensor(labels, dtype=torch.long)

In [8]:
print(vectors.shape)
print(labels.shape)
print(labels)

torch.Size([11, 26])
torch.Size([11])
tensor([1, 1, 1, 3, 3, 3, 2, 2, 2, 2, 3])


In [9]:
print(torch.max(vectors))

tensor(1.)


In [10]:
input_dim=len(vocabulary)
hidden_dim=50
output_dim=num_classes

class SimpleClassifier(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim):
    super().__init__() # runs the constructor of the parent class, nn.Module
    self.fc1=nn.Linear(input_dim, hidden_dim)
    self.relu=nn.ReLU()
    self.fc2=nn.Linear(hidden_dim, output_dim)
    # No final softmax layer : Pytorch's CrossEntropyLoss combines softmax and cross entropy loss internally. 

  def forward(self, x):
    x=self.fc1(x)
    x=self.relu(x)
    x=self.fc2(x)
    return x

model=SimpleClassifier(input_dim,hidden_dim,output_dim)

In [11]:
print(model)

SimpleClassifier(
  (fc1): Linear(in_features=26, out_features=50, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=50, out_features=11, bias=True)
)


In [12]:
print("W1.shape = ", model.fc1.weight.shape)   # shape (50, 26)
print("b1.shape = ", model.fc1.bias.shape)     # shape (50,)
print("W2.shape = ", model.fc2.weight.shape)   # shape (11, 50)
print("b2.shape = ", model.fc2.bias.shape)     # shape (11,)
print("b2 = ", model.fc2.bias)

W1.shape =  torch.Size([50, 26])
b1.shape =  torch.Size([50])
W2.shape =  torch.Size([11, 50])
b2.shape =  torch.Size([11])
b2 =  Parameter containing:
tensor([ 0.0657, -0.1207, -0.0709,  0.0096,  0.1117,  0.0745, -0.1161,  0.0157,
        -0.0834,  0.0061, -0.1178], requires_grad=True)


In [13]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(model.parameters(), lr=0.001)

num_steps = 3000

for step in range(num_steps):
  optimizer.zero_grad()
  loss=criterion(model(vectors), labels)
  loss.backward()
  optimizer.step()
  if step % 100 == 0:
      print(step)

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900


In [14]:
print(type(model.parameters()))
print(model.parameters())

<class 'generator'>
<generator object Module.parameters at 0x0000014AD96791C0>


In [15]:
new_docs=["listening to rock music is fun", "I love science very much", "cinema is the best", "movies", "cinema", "cinematography", "science music cinema", "cinema", "music", "science", "maths", "math", "mathematics", "algebra"]
class_names=["cinema","music","science"]

In [16]:
new_doc_vectors=torch.tensor([doc_to_bow(new_doc, vocabulary) for new_doc in new_docs], dtype=torch.float32)

In [17]:
print(new_doc_vectors.shape)

torch.Size([14, 26])


In [18]:
with torch.no_grad():
  outputs=model(new_doc_vectors)
  predicted_ids=torch.argmax(outputs,dim=1)
  print(predicted_ids)

for i, new_doc in enumerate(new_docs):
  print(f'{new_doc}:{class_names[predicted_ids[i].item()-1]}')

tensor([2, 3, 3, 1, 3, 3, 2, 3, 2, 3, 3, 3, 3, 3])
listening to rock music is fun:music
I love science very much:science
cinema is the best:science
movies:cinema
cinema:science
cinematography:science
science music cinema:music
cinema:science
music:music
science:science
maths:science
math:science
mathematics:science
algebra:science


In [19]:
new_doc = "algebra"
new_doc_vector=torch.tensor([doc_to_bow(new_doc, vocabulary)], dtype=torch.float32)
print(doc_to_bow(new_doc, vocabulary))
print(new_doc_vector)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0.]])
